In [4]:
import pandas as pd
from sqlalchemy import create_engine
from config import FEATURED_PATH, PG_CONFIG
 

In [5]:
conn_str = (
    f"postgresql+psycopg2://{PG_CONFIG['user']}:{PG_CONFIG['password']}"
    f"@{PG_CONFIG['host']}:{PG_CONFIG['port']}/{PG_CONFIG['database']}"
)
engine = create_engine(conn_str)

In [6]:
from sqlalchemy.types import Date

# --- Source 2: DHIS2 routine indicator data (ANC 1st visit) ---
dhis2_df = pd.read_csv("dhis2_indicator_values.csv")

dhis2_df = dhis2_df.rename(columns={
    "dx": "data_element_id",
    "ou": "org_unit_id",
    "pe": "period",
    "value": "anc1_visits",
})

dhis2_df["period"] = dhis2_df["period"].astype(str)
dhis2_df["anc1_visits"] = dhis2_df["anc1_visits"].astype(int)

month_start = pd.to_datetime(dhis2_df["period"], format="%Y%m")
dhis2_df["visit_month"] = month_start.dt.to_period("M").astype(str)
dhis2_df["visit_month_date"] = month_start.dt.date

dhis2_df = dhis2_df.sort_values("visit_month_date").reset_index(drop=True)

dhis2_df.to_sql(
    "dhis2_anc1_indicator", engine,
    if_exists="replace", index=False,
    dtype={"visit_month_date": Date()},
)
print(f"Loaded {len(dhis2_df)} rows into 'dhis2_anc1_indicator'")

# Quick check
pd.read_sql("SELECT * FROM dhis2_anc1_indicator ORDER BY visit_month_date", engine)

Loaded 11 rows into 'dhis2_anc1_indicator'


,data_element_id,org_unit_id,period,anc1_visits,visit_month,visit_month_date
0,fbfJHSPpUQD,fdc6uOvgoji,202510,1379,2025-10,2025-10-01
1,fbfJHSPpUQD,fdc6uOvgoji,202512,1410,2025-12,2025-12-01
2,fbfJHSPpUQD,fdc6uOvgoji,202601,1523,2026-01,2026-01-01
3,fbfJHSPpUQD,fdc6uOvgoji,202602,1387,2026-02,2026-02-01
4,fbfJHSPpUQD,fdc6uOvgoji,202603,1380,2026-03,2026-03-01
5,fbfJHSPpUQD,fdc6uOvgoji,202604,1566,2026-04,2026-04-01
6,fbfJHSPpUQD,fdc6uOvgoji,202605,2122,2026-05,2026-05-01
7,fbfJHSPpUQD,fdc6uOvgoji,202606,1718,2026-06,2026-06-01
8,fbfJHSPpUQD,fdc6uOvgoji,202607,1599,2026-07,2026-07-01
9,fbfJHSPpUQD,fdc6uOvgoji,202608,1624,2026-08,2026-08-01
